In [20]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings('ignore')

In [21]:
df = pd.read_excel(r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\Preprocessing\100_Pre_done_Combined.xlsx")

df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

df.head()

,Material,SLoc,Quantity,Pstng Date,order,Equipment,Technician name,Year,Tavg,Tmax,Tmin,RH,Month,Season,Delta_T,Region,Location
0,100,5023,-1,2020-01-04,48550533,10930429,Anil Sharma,2020,12.94,21.27,7.26,58.52,1,Winter,14.01,North1,Haryana
1,100,5024,-1,2020-01-06,48556766,10844557,Jogendra Singh,2020,14.94,22.58,9.03,56.47,1,Winter,13.55,North1,Jaipur
2,100,5030,-1,2020-01-06,48550093,10517828,Himanshu Kushwaha,2020,13.70,21.92,8.25,55.23,1,Winter,13.67,North1,Delhi
3,100,5002,-1,2020-01-06,48550185,10519283,Prasad Chokhat,2020,22.83,30.49,16.34,68.83,1,Winter,14.15,West1,Navi Mumbai
4,100,5044,-1,2020-01-06,48554032,10836205,Joby Varghese,2020,28.71,32.68,25.08,68.80,1,Winter,7.60,South,Cochin


In [22]:
monthly = (
    df.groupby(
        [
            pd.Grouper(key='Pstng Date', freq='MS'),
            'SLoc'
        ]
    )
    .agg(
        Consumption=('Quantity', lambda x: abs(x.sum())),
        Tavg=('Tavg', 'mean'),
        RH=('RH', 'mean'),
        Delta_T=('Delta_T', 'mean'),
        Region=('Region', 'first'),
        Location=('Location', 'first'),
        Season=('Season', 'first')
    )
    .reset_index()
)

monthly.head()

,Pstng Date,SLoc,Consumption,Tavg,RH,Delta_T,Region,Location,Season
0,2020-01-01,5001,19,21.661579,67.084211,13.224737,West1,Mumbai,Winter
1,2020-01-01,5002,4,22.910000,67.287500,11.530000,West1,Navi Mumbai,Winter
2,2020-01-01,5005,1,18.910000,54.610000,12.240000,West2,Ahmedabad,Winter
3,2020-01-01,5007,1,18.770000,74.430000,13.940000,West2,Raipur,Winter
4,2020-01-01,5010,2,21.490000,63.915000,14.455000,West2,Surat,Winter


In [23]:
# ============================================================
# MONTHLY WEATHER CLIMATOLOGY
# ============================================================

weather_climatology = (
    monthly.groupby(
        ['Location', 'Month']
    )
    .agg(
        Tavg=('Tavg', 'mean'),
        RH=('RH', 'mean'),
        Delta_T=('Delta_T', 'mean')
    )
    .reset_index()
)

print(weather_climatology.head())

KeyError: 'Month'

In [ ]:
# ============================================================
# FUTURE MONTHS
# ============================================================

forecast_horizon = 6

last_date = monthly['Pstng Date'].max()

future_dates = pd.date_range(
    start=last_date + pd.offsets.MonthBegin(1),
    periods=forecast_horizon,
    freq='MS'
)

future_dates

DatetimeIndex(['2026-05-01', '2026-06-01', '2026-07-01', '2026-08-01',
               '2026-09-01', '2026-10-01'],
              dtype='datetime64[us]', freq='MS')

In [ ]:
# ============================================================
# FUTURE SLOC FRAME
# ============================================================

sloc_master = (
    monthly[
        ['SLoc', 'Location', 'Region']
    ]
    .drop_duplicates()
)

future_rows = []

for dt in future_dates:

    temp = sloc_master.copy()

    temp['Pstng Date'] = dt

    future_rows.append(temp)

future_df = pd.concat(
    future_rows,
    ignore_index=True
)

future_df.head()

,SLoc,Location,Region,Pstng Date
0,5001,Mumbai,West1,2026-05-01
1,5002,Navi Mumbai,West1,2026-05-01
2,5003,Pune,West1,2026-05-01
3,5005,Ahmedabad,West2,2026-05-01
4,5010,Surat,West2,2026-05-01


In [ ]:
# ============================================================
# DATE FEATURES
# ============================================================

future_df['Year'] = future_df['Pstng Date'].dt.year
future_df['Month'] = future_df['Pstng Date'].dt.month

future_df['sin_month'] = np.sin(
    2*np.pi*future_df['Month']/12
)

future_df['cos_month'] = np.cos(
    2*np.pi*future_df['Month']/12
)

In [ ]:
# ============================================================
# SEASON MAPPING
# ============================================================

def month_to_season(m):

    if m in [12,1,2]:
        return 'Winter'

    elif m in [3,4,5]:
        return 'Summer'

    elif m in [6,7,8,9]:
        return 'Monsoon'

    else:
        return 'PostMonsoon'

future_df['Season'] = (
    future_df['Month']
    .apply(month_to_season)
)

In [ ]:
# ============================================================
# FUTURE WEATHER ASSUMPTION
# ============================================================

future_df = future_df.merge(
    weather_climatology,
    on=['Location', 'Month'],
    how='left'
)

future_df.head()

,SLoc,Location,Region,Pstng Date,Year,Month,sin_month,cos_month,Season,Tavg,RH,Delta_T
0,5001,Mumbai,West1,2026-05-01,2026,5,0.5,-0.866025,Summer,30.724141,65.951995,9.789407
1,5002,Navi Mumbai,West1,2026-05-01,2026,5,0.5,-0.866025,Summer,30.510893,67.566964,9.193393
2,5003,Pune,West1,2026-05-01,2026,5,0.5,-0.866025,Summer,27.954167,65.223958,12.472500
3,5005,Ahmedabad,West2,2026-05-01,2026,5,0.5,-0.866025,Summer,34.855000,38.632250,14.682583
4,5010,Surat,West2,2026-05-01,2026,5,0.5,-0.866025,Summer,32.182718,61.027579,11.874325


In [ ]:
monthly['Year'] = monthly['Pstng Date'].dt.year
monthly['Month'] = monthly['Pstng Date'].dt.month

monthly['sin_month'] = np.sin(
    2*np.pi*monthly['Month']/12
)

monthly['cos_month'] = np.cos(
    2*np.pi*monthly['Month']/12
)

In [ ]:
# ============================================================
# LATEST HISTORY
# ============================================================

latest_history = (
    monthly
    .sort_values('Pstng Date')
    .groupby('SLoc')
    .tail(12)
)

latest_history.head()

,Pstng Date,SLoc,Consumption,Tavg,RH,Delta_T,Region,Location,Season,Year,Month,sin_month,cos_month,lag_1,lag_2,lag_3,lag_6,lag_12,roll_3,roll_6
53,2021-11-01,5003,1,22.070000,81.150000,11.570000,West1,Pune,Post-Monsoon,2021,11,-5.000000e-01,0.866025,2.0,2.0,0.0,3.0,0.0,1.333333,1.500000
163,2021-12-01,5023,5,13.711111,49.325556,14.421111,North1,Haryana,Winter,2021,12,-2.449294e-16,1.000000,8.0,5.0,9.0,3.0,4.0,7.333333,6.833333
268,2021-12-01,5043,9,24.676667,82.886667,4.703333,South,Chennai,Winter,2021,12,-2.449294e-16,1.000000,4.0,9.0,6.0,2.0,7.0,6.333333,4.833333
54,2021-12-01,5003,2,19.605000,77.202500,11.995000,West1,Pune,Winter,2021,12,-2.449294e-16,1.000000,1.0,2.0,2.0,1.0,1.0,1.666667,1.166667
280,2021-12-01,5044,3,27.273333,79.903333,4.696667,South,Cochin,Winter,2021,12,-2.449294e-16,1.000000,1.0,2.0,0.0,1.0,2.0,1.000000,1.000000


In [ ]:
monthly = monthly.sort_values(
    ['SLoc','Pstng Date']
)

for lag in [1,2,3,6,12]:

    monthly[f'lag_{lag}'] = (
        monthly
        .groupby('SLoc')['Consumption']
        .shift(lag)
    )

In [ ]:
monthly['roll_3'] = (
    monthly
    .groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1).rolling(3).mean()
    )
)

monthly['roll_6'] = (
    monthly
    .groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1).rolling(6).mean()
    )
)

In [ ]:
monthly = monthly.dropna().reset_index(drop=True)

monthly.shape

(849, 20)

In [ ]:
split_date = '2025-01-01'

train = monthly[
    monthly['Pstng Date'] < split_date
]

val = monthly[
    monthly['Pstng Date'] >= split_date
]

In [ ]:
features = [
    'SLoc',
    'Region',
    'Location',
    'Season',

    'Tavg',
    'RH',
    'Delta_T',

    'Month',
    'Year',

    'sin_month',
    'cos_month',

    'lag_1',
    'lag_2',
    'lag_3',
    'lag_6',
    'lag_12',

    'roll_3',
    'roll_6'
]

target = 'Consumption'

In [ ]:
cat_features = [
    'SLoc',
    'Region',
    'Location',
    'Season'
]

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    loss_function='RMSE',
    verbose=100
)

model.fit(
    train[features],
    train[target],
    cat_features=cat_features
)

0:	learn: 23.7455578	total: 203ms	remaining: 3m 22s
100:	learn: 7.2295209	total: 6.55s	remaining: 58.3s
200:	learn: 4.7124528	total: 13.3s	remaining: 53s
300:	learn: 3.7304082	total: 19.8s	remaining: 46s
400:	learn: 3.0647180	total: 26.5s	remaining: 39.7s
500:	learn: 2.5261586	total: 33.1s	remaining: 32.9s
600:	learn: 2.1707542	total: 39.7s	remaining: 26.4s
700:	learn: 1.8926255	total: 46.4s	remaining: 19.8s
800:	learn: 1.6371688	total: 53s	remaining: 13.2s
900:	learn: 1.4250161	total: 59.5s	remaining: 6.54s
999:	learn: 1.2555739	total: 1m 6s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.03, loss_function='RMSE', verbose=100)

In [ ]:
val_pred = model.predict(
    val[features]
)

mae = mean_absolute_error(
    val[target],
    val_pred
)

rmse = np.sqrt(
    mean_squared_error(
        val[target],
        val_pred
    )
)

mape = np.mean(
    np.abs(
        (
            val[target]
            - val_pred
        )
        /
        val[target]
    )
) * 100

print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

MAE : 5.440987785464996
RMSE: 11.326271240903717
MAPE: inf


In [ ]:
forecast_pivot = forecast_df.pivot(
    index='SLoc',
    columns='Month_Name',
    values='Forecast'
)

forecast_pivot.to_excel(
    'SLoc_Forecast.xlsx'
)

NameError: name 'forecast_df' is not defined